In [2]:
# ============================================
# CELDA 1: CARGAR DATOS PROCESADOS
# ============================================

import pandas as pd
import numpy as np
import joblib
import sys
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate

# Cargar CSV con TODAS las features ya creadas por 03_eda_features.ipynb
df_train = pd.read_csv('../data/processed/consumo_granada_modelo.csv')
df_test = pd.read_csv('../data/raw/consumo_granada.csv')

print(f"📊 Dataset de entrenamiento cargado: {df_train.shape}")
print(f"📊 Dataset de test cargado: {df_test.shape}")
print(f"Features: {len(df_train.columns)-1}, Target: consumption_kwh")

# Separar Features (X) y Target (y) para entrenamiento
X_train = df_train.drop('consumption_kwh', axis=1)
y_train = df_train['consumption_kwh']

# Separar Features (X) y Target (y) para test
X_test = df_test.drop('consumption_kwh', axis=1)
y_test = df_test['consumption_kwh']

print(f"\n✅ Datos preparados:")
print(f"   - Entrenamiento: {X_train.shape}")
print(f"   - Test: {X_test.shape}")

📊 Dataset de entrenamiento cargado: (1947461, 35)
📊 Dataset de test cargado: (1947461, 5)
Features: 34, Target: consumption_kwh

✅ Datos preparados:
   - Entrenamiento: (1947461, 34)
   - Test: (1947461, 4)


In [5]:
# ============================================
# CELDA 2: ENTRENAR RANDOM FOREST
# ============================================

# Definir modelo
rf_model = RandomForestRegressor(
    n_estimators=300,        # Número de árboles ajustado para mayor estabilidad
    random_state=42,         # Semilla para reproducibilidad
    n_jobs=-1,               # Uso de todos los núcleos disponibles
    max_depth=25,            # Limitar la profundidad de los árboles para capturar relaciones complejas
    min_samples_split=5,     # Más muestras para dividir un nodo
    min_samples_leaf=2,      # Reducir el tamaño mínimo de las hojas
    max_features='sqrt',     # Considerar solo una fracción de las características en cada división
    max_samples=None         # Usar el 100% de los datos para cada árbol
)

print("🔧 Entrenando Random Forest con 5-Fold Cross-Validation...\n")

# Configurar métricas
scoring = {
    'MAE': 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error'
}

# Entrenar con CV
cv_results = cross_validate(rf_model, X_train, y_train, cv=5, scoring=scoring)

# Resultados
mae = -cv_results['test_MAE'].mean()
rmse = -cv_results['test_RMSE'].mean()

print(f"📈 Resultados Cross-Validation (5-Fold):")
print(f"   MAE:  {mae:.2f} kWh")
print(f"   RMSE: {rmse:.2f} kWh")

# Entrenar modelo final
print(f"\n🚀 Entrenando modelo final con {len(X_train)} muestras...")
rf_model.fit(X_train, y_train)

print("✅ Modelo Random Forest entrenado")


🔧 Entrenando Random Forest con 5-Fold Cross-Validation...

📈 Resultados Cross-Validation (5-Fold):
   MAE:  131.97 kWh
   RMSE: 196.36 kWh

🚀 Entrenando modelo final con 1947461 muestras...
✅ Modelo Random Forest entrenado


In [7]:
from sklearn.preprocessing import StandardScaler

# ============================================
# PREPROCESAMIENTO DE LOS DATOS
# ============================================

# Función para realizar la limpieza de datos
def clean_data(X):
    X_processed = X.copy()
    
    # 1. Convertir 'timestamp' a datetime
    if 'timestamp' in X_processed.columns:
        X_processed['timestamp'] = pd.to_datetime(X_processed['timestamp'], errors='coerce', utc=True)
    
    # 2. Limpiar nombres de zona
    if 'zone_name' in X_processed.columns:
        X_processed['zone_name'] = X_processed['zone_name'].str.strip()
        X_processed['zone_name'] = X_processed['zone_name'].str.replace(' ', '_')
        X_processed['zone_name_upper'] = X_processed['zone_name'].str.upper()
        X_processed = X_processed.drop(columns=['zone_name'], errors='ignore')
    
    # 3. Interpolación de valores faltantes
    if 'consumption_kwh' in X_processed.columns and 'temperature' in X_processed.columns and 'zone_name_upper' in X_processed.columns:
        X_processed['consumption_kwh'] = X_processed['consumption_kwh'].replace(0.0, np.nan)
        X_processed = X_processed.sort_values(['zone_name_upper', 'timestamp'])
        for col in ['consumption_kwh', 'temperature']:
            X_processed[col] = X_processed.groupby('zone_name_upper')[col].transform(
                lambda x: x.interpolate(method='linear')
            )
    
    return X_processed

# Función para realizar el feature engineering
def feature_engineering(X):
    X_processed = X.copy()
    
    if 'timestamp' in X_processed.columns:
        ts = X_processed['timestamp']
        
        # 1. Componentes temporales
        X_processed['hour'] = ts.dt.hour
        X_processed['month'] = ts.dt.month
        X_processed['day_of_week'] = ts.dt.dayofweek
        X_processed['day_of_month'] = ts.dt.day
        
        # 2. Transformaciones cíclicas
        X_processed['hour_sin'] = np.sin(2 * np.pi * X_processed['hour'] / 24)
        X_processed['hour_cos'] = np.cos(2 * np.pi * X_processed['hour'] / 24)
        X_processed['month_sin'] = np.sin(2 * np.pi * X_processed['month'] / 12)
        X_processed['month_cos'] = np.cos(2 * np.pi * X_processed['month'] / 12)
        
        # 3. Variables de calendario
        X_processed['is_weekend'] = X_processed['day_of_week'].isin([5, 6]).astype(int)
        X_processed['is_holiday'] = X_processed['timestamp'].apply(lambda x: 1 if (x.month, x.day) in [
            (1, 1), (6, 1), (28, 2), (1, 5), (15, 8), (12, 10), (1, 11), (6, 12), (8, 12), (25, 12)
        ] else 0)
        X_processed['is_non_working'] = ((X_processed['is_weekend'] == 1) | (X_processed['is_holiday'] == 1)).astype(int)
        
        # 4. Transformaciones climáticas
        if 'temperature' in X_processed.columns:
            X_processed['temp_sq'] = X_processed['temperature'] ** 2
        
        # 5. Rellenar NaNs en features derivadas con 0
        cols_to_fill_zero = [
            'hour', 'month', 'day_of_week', 'day_of_month', 'is_weekend', 
            'is_holiday', 'is_non_working', 'temp_sq', 'hour_sin', 'hour_cos', 
            'month_sin', 'month_cos'
        ]
        for col in cols_to_fill_zero:
            if col in X_processed.columns:
                X_processed[col] = X_processed[col].fillna(0)
    
    # 6. Eliminar columnas innecesarias
    columnas_a_borrar = ['timestamp', 'zone_id', 'fecha', 'hora', 'consumption_kwh']
    X_processed = X_processed.drop(columns=[col for col in columnas_a_borrar if col in X_processed.columns], errors='ignore')
    
    return X_processed

# Función para estandarizar y preparar los datos finales
def prepare_data(X_train, X_test):
    # Asegurar que las columnas de X_test_cleaned coincidan con las de X_train_cleaned
    missing_cols = set(X_train.columns) - set(X_test.columns)
    for col in missing_cols:
        X_test[col] = 0  # Agregar columnas faltantes con valor 0

    extra_cols = set(X_test.columns) - set(X_train.columns)
    if extra_cols:
        X_test = X_test.drop(columns=extra_cols)  # Eliminar columnas adicionales

    # Reordenar las columnas de X_test_cleaned para que coincidan con X_train_cleaned
    X_test = X_test[X_train.columns]

    # Estandarizar las columnas numéricas
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

    return X_train_scaled, X_test_scaled

# Preprocesamiento de los datos
print("🔄 Aplicando limpieza y feature engineering a los datos de entrenamiento...")
X_train_cleaned = clean_data(X_train)
X_train_cleaned = feature_engineering(X_train_cleaned)

print("🔄 Aplicando limpieza y feature engineering a los datos de prueba...")
X_test_cleaned = clean_data(X_test)
X_test_cleaned = feature_engineering(X_test_cleaned)

# Eliminar filas con valores NaN en X_train_cleaned y X_test_cleaned
print("🔄 Eliminando filas con valores NaN...")
X_train_cleaned = X_train_cleaned.dropna()
X_test_cleaned = X_test_cleaned.dropna()

# Preparar los datos finales
print("🔄 Preparando los datos finales...")
X_train_final, X_test_final = prepare_data(X_train_cleaned, X_test_cleaned)

print("✅ Preprocesamiento completado.")

🔄 Aplicando limpieza y feature engineering a los datos de entrenamiento...
🔄 Aplicando limpieza y feature engineering a los datos de prueba...
🔄 Eliminando filas con valores NaN...
🔄 Preparando los datos finales...
✅ Preprocesamiento completado.


In [11]:
# ============================================
# EVALUACIÓN DEL MODELO
# ============================================

# Asegurar que las columnas de X_test_cleaned coincidan con las de X_train_cleaned
print("🔄 Asegurando que las características de X_test_cleaned coincidan con las de X_train_cleaned...")
missing_cols = set(X_train_cleaned.columns) - set(X_test_cleaned.columns)
for col in missing_cols:
    X_test_cleaned[col] = 0  # Agregar columnas faltantes con valor 0

extra_cols = set(X_test_cleaned.columns) - set(X_train_cleaned.columns)
if extra_cols:
    X_test_cleaned = X_test_cleaned.drop(columns=extra_cols)  # Eliminar columnas adicionales

# Reordenar las columnas de X_test_cleaned para que coincidan con X_train_cleaned
X_test_cleaned = X_test_cleaned[X_train_cleaned.columns]

# Alinear índices entre X_test_cleaned y y_test
print("🔄 Alineando índices entre X_test_cleaned y y_test...")
aligned_indices = X_test_cleaned.index.intersection(y_test.index)
X_test_cleaned = X_test_cleaned.loc[aligned_indices]
y_test = y_test.loc[aligned_indices]

# Verificar que X_test_cleaned y y_test tengan el mismo número de filas
if len(X_test_cleaned) != len(y_test):
    raise ValueError(
        f"Inconsistencia detectada: X_test_cleaned tiene {len(X_test_cleaned)} filas, "
        f"pero y_test tiene {len(y_test)} filas."
    )

print(f"\n🧪 Evaluando en test set ({len(X_test_cleaned)} muestras)...")

# Realizar predicción
y_pred = rf_model.predict(X_test_cleaned)

# Calcular métricas
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae_test = mean_absolute_error(y_test, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))

# Mostrar métricas
print(f"\n📊 Métricas en Test:")
print(f"   MAE:  {mae_test:.2f} kWh")
print(f"   RMSE: {rmse_test:.2f} kWh")

print("\n✅ Evaluación completada.")

🔄 Asegurando que las características de X_test_cleaned coincidan con las de X_train_cleaned...
🔄 Alineando índices entre X_test_cleaned y y_test...

🧪 Evaluando en test set (1931819 muestras)...

📊 Métricas en Test:
   MAE:  763.07 kWh
   RMSE: 1138.58 kWh

✅ Evaluación completada.


In [14]:
# ============================================
# CELDA 3: GUARDAR SOLO EL MODELO
# ============================================

# Guardar SOLO el modelo entrenado
os.makedirs("../data/models", exist_ok=True)
model_path = "../data/models/random_forest_model.joblib"
joblib.dump(rf_model, model_path)

print(f"💾 Modelo guardado en: {model_path}")
print(f"   Tamaño: {os.path.getsize(model_path) / (1024*1024):.2f} MB")

print("\n✅ Modelo listo")
print("   NOTA: La API usará un pipeline para transformar datos RAW antes de usar este modelo")

💾 Modelo guardado en: ../data/models/random_forest_model.joblib
   Tamaño: 3105.61 MB

🧪 Evaluando en test set (1947461 muestras)...


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- timestamp
- zone_id
- zone_name
Feature names seen at fit time, yet now missing:
- day_of_month
- day_of_week
- hour
- hour_cos
- hour_sin
- ...


# 🔍 DIAGNÓSTICO: Feature Importance

Analizar qué features están dominando el modelo y por qué no captura la variabilidad horaria.

In [6]:
# ============================================
# ANÁLISIS DE IMPORTANCIA DE FEATURES
# ============================================

import matplotlib.pyplot as plt

# Obtener importancias de features
feature_importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("📊 TOP 15 FEATURES MÁS IMPORTANTES:\n")
print(feature_importances.head(15).to_string(index=False))

# Visualizar las top 15
plt.figure(figsize=(10, 6))
plt.barh(feature_importances.head(15)['feature'], feature_importances.head(15)['importance'])
plt.xlabel('Importancia')
plt.title('Top 15 Features Más Importantes - Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Analizar features temporales específicamente
temporal_features = ['hour', 'hour_sin', 'hour_cos', 'month', 'month_sin', 'month_cos', 
                     'day_of_week', 'day_of_month', 'is_weekend', 'is_holiday', 'is_non_working']
temporal_importance = feature_importances[feature_importances['feature'].isin(temporal_features)]

print(f"\n🕐 IMPORTANCIA DE FEATURES TEMPORALES:")
print(temporal_importance.to_string(index=False))
print(f"\n   Suma total de importancia temporal: {temporal_importance['importance'].sum():.4f}")

# Analizar features de zona
zone_features = [col for col in X_train.columns if col.startswith('zona_')]
zone_importance = feature_importances[feature_importances['feature'].isin(zone_features)]

print(f"\n📍 IMPORTANCIA DE FEATURES DE ZONA:")
print(f"   Suma total de importancia de zonas: {zone_importance['importance'].sum():.4f}")
print(f"   Top 5 zonas más importantes:")
print(zone_importance.head(5).to_string(index=False))

NameError: name 'rf_model' is not defined

# 🧪 TEST: Predicción Horaria (30/12/2025 - Pts Tecnologico)

Simular predicciones hora por hora para **verificar si el modelo captura variabilidad**.

In [ ]:
# ============================================
# TEST DE VARIABILIDAD HORARIA
# ============================================

# Verificar que el modelo esté entrenado
if 'rf_model' not in globals():
    print("❌ ERROR: Modelo no encontrado.")
    print("💡 Por favor, ejecuta primero la CELDA 2 para entrenar el modelo.")
else:
    # Cargar datos con FEATURES ya creadas
    df_modelo = pd.read_csv('../data/processed/consumo_granada_modelo.csv')
    df_cleaned = pd.read_csv('../data/processed/consumo_granada_cleaned.csv')
    df_cleaned['timestamp'] = pd.to_datetime(df_cleaned['timestamp'])

    # Filtrar datos reales de un día específico en Pts Tecnologico
    test_date = '2024-12-11'
    test_zone = 'Pts_Tecnologico'

    # Obtener índices del día de prueba en el CSV limpio
    mask_cleaned = (df_cleaned['timestamp'].dt.date == pd.to_datetime(test_date).date()) & \
                   (df_cleaned['zone_name'] == test_zone)
    df_test_day = df_cleaned[mask_cleaned].copy()

    if len(df_test_day) > 0:
        print(f"📅 Datos reales encontrados: {len(df_test_day)} registros para {test_date} en {test_zone}\n")
        
        # Obtener las features correspondientes del CSV de modelo
        # Usar los mismos índices para buscar en consumo_granada_modelo.csv
        test_indices = df_test_day.index
        
        # Filtrar features del modelo usando los índices
        X_test_features = df_modelo.loc[test_indices].drop('consumption_kwh', axis=1)
        y_test_real = df_modelo.loc[test_indices]['consumption_kwh']
        
        # Predecir con el modelo
        y_pred = rf_model.predict(X_test_features)
        
        # Crear DataFrame de comparación
        df_comparison = pd.DataFrame({
            'hora': df_test_day['timestamp'].dt.hour.values,
            'temperatura_real': df_test_day['temperature'].values,
            'consumo_real': y_test_real.values,
            'consumo_predicho': y_pred,
            'error_abs': np.abs(y_test_real.values - y_pred)
        }).sort_values('hora')
        
        print("🔍 COMPARACIÓN HORA POR HORA:\n")
        print(df_comparison.to_string(index=False))
        
        # Estadísticas
        print(f"\n📊 ESTADÍSTICAS:")
        print(f"   MAE (este día): {df_comparison['error_abs'].mean():.2f} kWh")
        print(f"   Consumo real promedio: {df_comparison['consumo_real'].mean():.2f} kWh")
        print(f"   Consumo predicho promedio: {df_comparison['consumo_predicho'].mean():.2f} kWh")
        print(f"   Variabilidad real (std): {df_comparison['consumo_real'].std():.2f} kWh")
        print(f"   Variabilidad predicha (std): {df_comparison['consumo_predicho'].std():.2f} kWh")
        
        # Gráfica comparativa
        plt.figure(figsize=(12, 6))
        plt.plot(df_comparison['hora'], df_comparison['consumo_real'], 'o-', label='Real', linewidth=2, markersize=8)
        plt.plot(df_comparison['hora'], df_comparison['consumo_predicho'], 's-', label='Predicho', linewidth=2, markersize=8)
        plt.xlabel('Hora del día', fontsize=12)
        plt.ylabel('Consumo (kWh)', fontsize=12)
        plt.title(f'Consumo Real vs Predicho - {test_date} - {test_zone}', fontsize=14)
        plt.legend(fontsize=11)
        plt.grid(True, alpha=0.3)
        plt.xticks(range(0, 24, 2))
        plt.tight_layout()
        plt.show()
        
        # Diagnóstico
        std_predicho = df_comparison['consumo_predicho'].std()
        std_real = df_comparison['consumo_real'].std()
        
        print(f"\n🔬 DIAGNÓSTICO:")
        print(f"   Ratio de variabilidad (predicho/real): {std_predicho/std_real:.2%}")
        
        if std_predicho < 50:
            print("\n⚠️ PROBLEMA CRÍTICO:")
            print("   Las predicciones son casi PLANAS (variabilidad < 50 kWh).")
            print("   El modelo NO está capturando el patrón horario correctamente.")
            print("\n💡 POSIBLES CAUSAS:")
            print("   1. Features temporales (hour, hour_sin, hour_cos) tienen baja importancia")
            print("   2. Features de zona dominan sobre la hora")
            print("   3. max_depth=25 está limitando el aprendizaje")
            print("\n🔧 SOLUCIONES:")
            print("   - Añadir features de interacción (hour × temperature, hour × zone)")
            print("   - Aumentar max_depth a None (sin límite)")
            print("   - Probar Gradient Boosting (mejor para relaciones no lineales)")
        elif std_predicho < std_real * 0.5:
            print("\n⚠️ PROBLEMA MODERADO:")
            print("   El modelo captura solo el 50% de la variabilidad horaria.")
            print("   Necesita mejorar para aplicaciones reales.")
        else:
            print("\n✅ VARIABILIDAD ACEPTABLE:")
            print("   El modelo está capturando el patrón horario correctamente.")
    else:
        print(f"❌ No se encontraron datos para {test_date} en {test_zone}")
        print(f"\n💡 Prueba con otra fecha disponible en tu CSV")

📅 Datos reales encontrados: 24 registros para 2024-12-11 en Pts_Tecnologico



NameError: name 'rf_model' is not defined